# Dimensionsreduktion — PCA

## Das Problem mit vielen Dimensionen

```
Stell dir vor: Ein Datensatz mit 100 Features.
→ Schwer zu visualisieren
→ Viele Features sind korreliert (redundant)
→ Modell wird langsam und kann overfitted werden

Lösung: Dimensionsreduktion — weniger Features, wichtigste Info bleibt!
```

**Methoden:**
- **PCA** (Principal Component Analysis) — heute
- t-SNE — für Visualisierung
- UMAP — moderne Alternative zu t-SNE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine, load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
%matplotlib inline

## 1. Varianz verstehen

PCA sucht die Richtungen mit der **größten Varianz**.
Warum Varianz? Weil Features mit hoher Varianz mehr Information enthalten!

```
Niedrige Varianz: [5, 5, 5, 5, 5] → Fast immer gleich → wenig Info
Hohe Varianz:     [1, 8, 2, 9, 3] → Große Unterschiede → viel Info
```

In [ ]:
# Wine Dataset laden — 13 Features, 178 Proben
wine_data = load_wine()
X = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
y = wine_data.target

print(f"Shape: {X.shape}  →  {X.shape[1]} Features!")
print(f"\nFeatures: {list(wine_data.feature_names)}")
X.head()

In [ ]:
# WICHTIG: Vor PCA immer skalieren!
# Ohne Skalierung dominieren Features mit großen Werten (z.B. proline ~1000)
# die Features mit kleinen Werten (z.B. hue ~1.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Vor Skalierung — Mittelwerte:")
print(X.mean().round(2))
print("\nNach Skalierung — Mittelwerte (alle ~0):")
print(pd.DataFrame(X_scaled, columns=wine_data.feature_names).mean().round(3))

## 2. PCA — Explained Variance

Zuerst PCA ohne Begrenzung der Komponenten → sehen wie viel Varianz jede Komponente erklärt.

In [ ]:
# PCA auf alle 13 Komponenten
pca_full = PCA()
pca_full.fit(X_scaled)

# Varianz pro Komponente
varianz = pca_full.explained_variance_ratio_ * 100

print("Erklärte Varianz pro Principal Component:")
for i, v in enumerate(varianz):
    print(f"  PC{i+1}: {v:.1f}%")

print(f"\nNur PC1 + PC2 = {varianz[0]+varianz[1]:.1f}% der gesamten Varianz")

In [ ]:
# Scree Plot — wie viele Komponenten brauche ich?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Links: Varianz pro Komponente
axes[0].bar(range(1, len(varianz)+1), varianz, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Erklärte Varianz (%)')
axes[0].set_title('Varianz pro Komponente (Scree Plot)')

# Rechts: Kumulierte Varianz
kum_varianz = np.cumsum(varianz)
axes[1].plot(range(1, len(kum_varianz)+1), kum_varianz, 'bo-', linewidth=2)
axes[1].axhline(y=80, color='red', linestyle='--', label='80% Schwelle')
axes[1].axhline(y=95, color='orange', linestyle='--', label='95% Schwelle')
axes[1].set_xlabel('Anzahl Komponenten')
axes[1].set_ylabel('Kumulierte Varianz (%)')
axes[1].set_title('Kumulierte erklärte Varianz')
axes[1].legend()

plt.tight_layout()
plt.show()

# Wie viele Komponenten für 80%?
for n, v in enumerate(kum_varianz):
    if v >= 80:
        print(f"→ {n+1} Komponenten erklären {v:.1f}% der Varianz (≥80%)")
        break

## 3. PCA anwenden

### Methode 1: Feste Komponentenanzahl

In [ ]:
# PCA auf 2 Komponenten — gut für Visualisierung
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

print(f"Original Shape:  {X_scaled.shape}   → {X_scaled.shape[1]} Features")
print(f"PCA Shape:       {X_pca.shape}    → {X_pca.shape[1]} Features")
print(f"Erklärte Varianz: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%")

# Visualisierung mit echten Labels
plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.8, s=60, edgecolors='white')
plt.colorbar(scatter, label='Weinsorte')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% Varianz)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% Varianz)')
plt.title('Wine Dataset — 13 Features → 2 PCA-Komponenten')
plt.show()

In [ ]:
# Methode 2: Varianz-Schwelle (empfohlen!)
# PCA(0.80) → so viele Komponenten wie nötig, um 80% Varianz zu erklären

pca_80 = PCA(n_components=0.80)
X_pca_80 = pca_80.fit_transform(X_scaled)

print(f"PCA mit 80% Varianz-Schwelle:")
print(f"  → {pca_80.n_components_} Komponenten (von 13)")
print(f"  → {pca_80.explained_variance_ratio_.sum()*100:.1f}% Varianz erhalten")

## 4. PCA in einer ML Pipeline

PCA als Vorverarbeitungsschritt vor dem Modell — alles in einer Pipeline!

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(wine_data.data, y, test_size=0.2, random_state=42)

# Pipeline: StandardScaler → PCA → Modell
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(0.80)),                    # 80% Varianz behalten
    ('model', DecisionTreeClassifier(random_state=42))
])

score_pca = pipe.fit(X_train, y_train).score(X_test, y_test)

# Zum Vergleich: ohne PCA
pipe_no_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier(random_state=42))
])
score_no_pca = pipe_no_pca.fit(X_train, y_train).score(X_test, y_test)

n_features_pca = pipe.named_steps['pca'].n_components_

print(f"Ohne PCA (13 Features): {score_no_pca:.2%}")
print(f"Mit PCA ({n_features_pca} Features):  {score_pca:.2%}")
print(f"\nFazit: {n_features_pca} Features statt 13 — Performance {'besser' if score_pca >= score_no_pca else 'ähnlich'}!")

## 5. PCA für Clustering-Visualisierung

Clustering auf hochdimensionalen Daten → Ergebnis schwer visualisierbar.
Lösung: K-Means auf Originaldaten, dann PCA für die Visualisierung!

In [ ]:
# K-Means auf skalierten Daten
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# PCA auf 2D für Visualisierung
pca_viz = PCA(n_components=2)
X_2d = pca_viz.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Links: K-Means Cluster
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7, s=60)
axes[0].set_title('K-Means Cluster (K=3) in PCA-Raum')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# Rechts: Echte Wein-Labels
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='viridis', alpha=0.7, s=60)
axes[1].set_title('Echte Wein-Labels in PCA-Raum')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

print(f"Silhouette Score K-Means: {silhouette_score(X_scaled, cluster_labels):.4f}")

## 6. Schrittweise PCA (Theorie-Demo)

Was passiert mathematisch hinter den Kulissen?
Hier auf dem Iris Dataset (4 Features → einfacher zu verstehen)

In [ ]:
# Iris Dataset (4 Features)
iris = load_iris()
X_iris = iris.data

# Schritt 1: Standardisieren
scaler_iris = StandardScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)
print("Schritt 1: Skaliert ✓")

# Schritt 2: Kovarianzmatrix
cov_matrix = np.cov(X_iris_scaled.T)
print(f"\nSchritt 2: Kovarianzmatrix ({cov_matrix.shape})")
print(cov_matrix.round(3))

# Schritt 3: Eigenwerte & Eigenvektoren
eigen_werte, eigen_vektoren = np.linalg.eig(cov_matrix)
print(f"\nSchritt 3: Eigenwerte: {eigen_werte.round(3)}")

# Schritt 4: Sortieren (größter zuerst)
idx = np.argsort(eigen_werte)[::-1]
eigen_werte_sorted = eigen_werte[idx]
eigen_vektoren_sorted = eigen_vektoren[:, idx]

varianz_erklaert = eigen_werte_sorted / eigen_werte_sorted.sum() * 100
print("\nSchritt 4: Sortierte erklärte Varianz:")
for i, v in enumerate(varianz_erklaert):
    print(f"  PC{i+1}: {v:.1f}%")

# Schritt 5: Top 2 auswählen & projizieren
W = eigen_vektoren_sorted[:, :2]  # 4x2 Projektionsmatrix
X_proj = X_iris_scaled.dot(W)     # 150x4 @ 4x2 = 150x2

print(f"\nSchritt 5+6: Projiziert auf {X_proj.shape[1]}D")

# Visualisierung
plt.figure(figsize=(9, 6))
plt.scatter(X_proj[:, 0], X_proj[:, 1], c=iris.target, cmap='viridis', alpha=0.8, s=60, edgecolors='white')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Iris Dataset — Manuelle PCA (4 → 2 Features)')
plt.colorbar(label='Iris-Art')
plt.show()

## Zusammenfassung

| Schritt | Code |
|---------|------|
| Skalieren | `StandardScaler().fit_transform(X)` |
| PCA anwenden | `PCA(n_components=0.80).fit_transform(X_scaled)` |
| Varianz checken | `pca.explained_variance_ratio_` |
| In Pipeline | `Pipeline([('scaler', ...), ('pca', ...), ('model', ...)])` |

**Goldene Regel:** Immer erst skalieren, dann PCA!

**Faustregel:** 80-95% erklärte Varianz ist ein guter Richtwert.